#1.字符串输出解析器 StrOutputParser

In [6]:
from xml.etree.ElementTree import XMLParser

from langchain_core.output_parsers import StrOutputParser, XMLOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv

#1.获取大模型
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.8,

)

#2.调用大模型
response = chat_model.invoke("什么是大预言模型")
print(type(response))
#3.如何获取一个字符串的输出结果

#方式1：自己调用输出结果的content
print(type(response.content))

#方式2：使用StrOutputOarser()
parser = StrOutputParser()
res = parser.invoke(response)
print(type(res))

<class 'langchain_core.messages.ai.AIMessage'>
<class 'str'>
<class 'str'>


#2.JSON解析器：JsonOutputParser

In [7]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个靠谱的{role}"),
    ("human", "{question}")
])

prompt=chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说?问题用q表示，答案用a表示，返回一个JSON格式的数据"})

response = chat_model.invoke(prompt)

#获取一个JsonOutputParser的实例
json_parser = JsonOutputParser()
res = json_parser.invoke(response)
print(res)

{'q': '人工智能用英文怎么说?', 'a': 'Artificial Intelligence'}


举例


In [8]:
from langchain_core.prompts import PromptTemplate
joke_query = "给我讲一个笑话"

parser = JsonOutputParser()
prompt_template = PromptTemplate.from_template(
    template="回答用户的查询\n 满足的格式为{format_instructions}\n 问题为{question}\n",
    partial_variables={"format_instructions":parser.get_format_instructions()},
)

prompt = prompt_template.invoke({"question":joke_query})

response = chat_model.invoke(prompt)
json_result = json_parser.invoke(response)
print(json_result)

{'joke': '为什么海洋总是那么快乐？因为它有很多浪！'}


In [9]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个靠谱的{role}"),
    ("human", "{question}")
])

# prompt=chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说?问题用q表示，答案用a表示，返回一个JSON格式的数据"})
#
# response = chat_model.invoke(prompt)

chain = chat_prompt_template | chat_model | json_parser
res = chain.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说?问题用q表示，答案用a表示，返回一个JSON格式的数据"})
print(res)

{'q': '人工智能用英文怎么说?', 'a': 'Artificial Intelligence'}


#3.XMLOutputParser XML输出解析器的使用

In [11]:
actoy_query = "周星驰"
response = chat_model.invoke(f"请生成{actoy_query}的简短电影记录，将影片附在<movie></movie>标签中")

print(response.content)

周星驰是中国著名的喜剧演员和导演，因其独特的喜剧风格而广受欢迎。他的电影作品通常充满幽默感和对社会现象的讽刺。以下是周星驰的一些代表性电影记录：

<movie>
    <title>大话西游之大圣娶亲</title>
    <year>1995</year>
    <genre>喜剧 / 爱情 / 奇幻</genre>
    <summary>讲述了孙悟空为爱而战的传奇故事，融入了大量的幽默元素和感人情节。</summary>
</movie>

<movie>
    <title>少林足球</title>
    <year>2001</year>
    <genre>喜剧 / 体育</genre>
    <summary>通过足球比赛展现了少林武术的魅力，结合幽默的情节和激情的比赛，受到观众的热烈欢迎。</summary>
</movie>

<movie>
    <title>功夫</title>
    <year>2004</year>
    <genre>动作 / 喜剧</genre>
    <summary>讲述了一名小混混在武术学校成长的故事，充满了创意的动作场面和搞笑元素。</summary>
</movie>

<movie>
    <title>西游降魔篇</title>
    <year>2013</year>
    <genre>奇幻 / 动作 / 喜剧</genre>
    <summary>重述了经典的西游记故事，加入了现代的喜剧风格，展现了唐僧与妖魔之间的斗智斗勇。</summary>
</movie>

周星驰的电影作品，既有搞笑的情节，又深刻地反映了人性和社会现象，让人捧腹大笑的同时，也引发了深思。


In [ ]:
xml_parser = XMLOutputParser()
print(xml_parser.get_format_instructions())

使用get_format_instructions()结构实现：

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core. output_parsers. xml import XMLOutputParser

actor_query = "生成汤姆 汉克斯的简短电影，用中文回复"

parser = XMLOutputParser()

prompt_template = PromptTemplate.from_template(
    template="用户的问题:{query} \n 使用的格式：{format_instructions}"
)

prompt_template2 = prompt_template.partial(format_instructions=parser.get_format_instructions())

prompt = prompt_template2.invoke({"query":actor_query})

response = chat_model.invoke(prompt)

print(response.content)

```xml
<电影>
   <标题>大白鲨</标题>
   <简介>汤姆·汉克斯在这部惊悚片中饰演一名海洋生物学家，他与一条巨大白鲨展开了一场惊心动魄的追逐战。</简介>
   <类型>惊悚/冒险</类型>
   <发行年份>2023</发行年份>
   <角色>
      <主角>汤姆·汉克斯</主角>
      <配角>其他角色信息</配角>
   </角色>
   <导演>某某导演</导演>
</电影>
```
